# Project 2

Using datasets from the NYC Department of Health and Mental Hygiene (DOHMH) and the MTA, I will explore how daily COVID-19 case counts relate to subway and bus ridership in New York City from 2020 onward.

The DOHMH dataset reports daily counts of confirmed COVID-19 cases, hospitalizations, and deaths. The MTA dataset reports estimated daily ridership for subways, buses, and other modes, as well as each mode’s ridership as a percentage of a comparable pre-pandemic day.

During the COVID-19 pandemic, public transit use became closely tied to perceived health risk. When reported COVID-19 cases rise, people may avoid shared spaces like subway cars and buses. When cases fall, ridership may slowly recover toward pre-pandemic levels.

Hypothesis:

Days with higher COVID-19 case counts will be associated with lower subway and bus ridership, especially when measured as a percentage of pre-pandemic levels.

To test this, I will merge the DOHMH daily case counts with MTA daily ridership data on the date, and then examine the relationships between COVID-19 case trends and subway/bus usage.


Data Sources

This project uses two publicly available datasets from NYC Open Data and New York State Open Data:

COVID-19 Daily Counts of Cases, Hospitalizations, and Deaths (DOHMH)
NYC Department of Health and Mental Hygiene
https://data.cityofnewyork.us/Health/COVID-19-Daily-Counts-of-Cases-Hospitalizations-an/rc75-m7u3/about_data

This dataset provides daily confirmed COVID-19 case counts, hospitalizations, and deaths for New York City, including 7-day rolling averages. It is updated regularly and covers the full period of the pandemic.

MTA Daily Ridership Data (2020–2025)
Metropolitan Transportation Authority
https://data.ny.gov/Transportation/MTA-Daily-Ridership-Data-2020-2025/vxuj-8kew/about_data

This dataset reports daily subway, bus, Access-A-Ride, LIRR, Metro-North, and bridge/tunnel traffic volumes, along with the percentage of pre-pandemic ridership for each mode.

1. Importing the datasets

In [2]:
import pandas as pd
import plotly.express as px
from IPython.display import HTML

# Load the COVID-19 daily counts dataset
covid = pd.read_csv('COVID-19_Daily_Counts_of_Cases,_Hospitalizations,_and_Deaths_20251121.csv')

# Load the MTA daily ridership dataset
mta = pd.read_csv('/Users/joyceyoyo/Desktop/Code/yaoyue.github.io/MTA_Daily_Ridership_Data__2020_-_2025_20251121.csv')

covid.head(), mta.head()


(  date_of_interest CASE_COUNT PROBABLE_CASE_COUNT HOSPITALIZED_COUNT  \
 0       02/29/2020          1                   0                  1   
 1       03/01/2020          0                   0                  1   
 2       03/02/2020          0                   0                  2   
 3       03/03/2020          1                   0                  7   
 4       03/04/2020          5                   0                  2   
 
    DEATH_COUNT CASE_COUNT_7DAY_AVG ALL_CASE_COUNT_7DAY_AVG  \
 0            0                   0                       0   
 1            0                   0                       0   
 2            0                   0                       0   
 3            0                   0                       0   
 4            0                   0                       0   
 
   HOSP_COUNT_7DAY_AVG  DEATH_COUNT_7DAY_AVG BX_CASE_COUNT  ... SI_CASE_COUNT  \
 0                   0                     0             0  ...             0   
 1                

The COVID dataset includes columns like CASE_COUNT, CASE_COUNT_7DAY_AVG, and date_of_interest.
The MTA dataset includes Date, total ridership numbers, and % of Comparable Pre-Pandemic Day for different modes.

2. Exploring the COVID-19 daily counts

First, I will inspect the COVID dataset’s structure.

In [3]:
covid.info()
covid[["date_of_interest", "CASE_COUNT", "CASE_COUNT_7DAY_AVG"]].head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2054 entries, 0 to 2053
Data columns (total 55 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   date_of_interest                 2054 non-null   object
 1   CASE_COUNT                       2054 non-null   object
 2   PROBABLE_CASE_COUNT              2054 non-null   object
 3   HOSPITALIZED_COUNT               2054 non-null   object
 4   DEATH_COUNT                      2054 non-null   int64 
 5   CASE_COUNT_7DAY_AVG              2054 non-null   object
 6   ALL_CASE_COUNT_7DAY_AVG          2054 non-null   object
 7   HOSP_COUNT_7DAY_AVG              2054 non-null   object
 8   DEATH_COUNT_7DAY_AVG             2054 non-null   int64 
 9   BX_CASE_COUNT                    2054 non-null   object
 10  BX_PROBABLE_CASE_COUNT           2054 non-null   object
 11  BX_HOSPITALIZED_COUNT            2054 non-null   int64 
 12  BX_DEATH_COUNT                   2

,date_of_interest,CASE_COUNT,CASE_COUNT_7DAY_AVG
0,02/29/2020,1,0
1,03/01/2020,0,0
2,03/02/2020,0,0
3,03/03/2020,1,0
4,03/04/2020,5,0


Next, I will convert the date column to a proper datetime type and the 7-day average to numeric.

In [4]:
covid_clean = covid.copy()

# Convert date to datetime
covid_clean["date"] = pd.to_datetime(covid_clean["date_of_interest"])

# Ensure 7-day average case counts are numeric
covid_clean["CASE_COUNT_7DAY_AVG"] = (
    covid_clean["CASE_COUNT_7DAY_AVG"]
      .astype(str)
      .str.replace(",", "", regex=False)
)

covid_clean["CASE_COUNT_7DAY_AVG"] = pd.to_numeric(
    covid_clean["CASE_COUNT_7DAY_AVG"],
    errors="coerce"
)

covid_clean[["date", "CASE_COUNT_7DAY_AVG"]].head()


,date,CASE_COUNT_7DAY_AVG
0,2020-02-29,0
1,2020-03-01,0
2,2020-03-02,0
3,2020-03-03,0
4,2020-03-04,0


3. Exploring the MTA ridership data

In [5]:
mta.info()
mta[[
    "Date",
    "Subways: % of Comparable Pre-Pandemic Day",
    "Buses: % of Comparable Pre-Pandemic Day"
]].head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1776 entries, 0 to 1775
Data columns (total 15 columns):
 #   Column                                                   Non-Null Count  Dtype 
---  ------                                                   --------------  ----- 
 0   Date                                                     1776 non-null   object
 1   Subways: Total Estimated Ridership                       1776 non-null   object
 2   Subways: % of Comparable Pre-Pandemic Day                1776 non-null   object
 3   Buses: Total Estimated Ridership                         1776 non-null   object
 4   Buses: % of Comparable Pre-Pandemic Day                  1776 non-null   object
 5   LIRR: Total Estimated Ridership                          1776 non-null   object
 6   LIRR: % of Comparable Pre-Pandemic Day                   1776 non-null   object
 7   Metro-North: Total Estimated Ridership                   1776 non-null   object
 8   Metro-North: % of Comparable Pre-Pande

,Date,Subways: % of Comparable Pre-Pandemic Day,Buses: % of Comparable Pre-Pandemic Day
0,03/01/2020,97%,99%
1,03/02/2020,96%,99%
2,03/03/2020,98%,99%
3,03/04/2020,99%,97%
4,03/05/2020,99%,100%


In [6]:
mta_clean = mta.copy()

# Convert date column
mta_clean["date"] = pd.to_datetime(mta_clean["Date"])

# Clean percentage columns: remove '%' and convert to numeric
for col in [
    "Subways: % of Comparable Pre-Pandemic Day",
    "Buses: % of Comparable Pre-Pandemic Day"
]:
    mta_clean[col] = (
        mta_clean[col]
          .astype(str)
          .str.replace("%", "", regex=False)
    )
    mta_clean[col] = pd.to_numeric(mta_clean[col], errors="coerce")

mta_clean[[
    "date",
    "Subways: % of Comparable Pre-Pandemic Day",
    "Buses: % of Comparable Pre-Pandemic Day"
]].head()


,date,Subways: % of Comparable Pre-Pandemic Day,Buses: % of Comparable Pre-Pandemic Day
0,2020-03-01,97,99
1,2020-03-02,96,99
2,2020-03-03,98,99
3,2020-03-04,99,97
4,2020-03-05,99,100


4. Merging the datasets on date

To relate COVID-19 case trends to transit usage, I will merge the two cleaned datasets on the date column.

In [7]:
combined = pd.merge(
    covid_clean[["date", "CASE_COUNT_7DAY_AVG"]],
    mta_clean[[
        "date",
        "Subways: % of Comparable Pre-Pandemic Day",
        "Buses: % of Comparable Pre-Pandemic Day"
    ]],
    on="date",
    how="inner"
)

combined = combined.sort_values("date")
combined.head()


,date,CASE_COUNT_7DAY_AVG,Subways: % of Comparable Pre-Pandemic Day,Buses: % of Comparable Pre-Pandemic Day
0,2020-03-01,0,97,99
1,2020-03-02,0,96,99
2,2020-03-03,0,98,99
3,2020-03-04,0,99,97
4,2020-03-05,0,99,100


5. Relationship 1: COVID-19 Cases and Subway Ridership

First, I will examine the relationship between COVID-19 cases and subway ridership.

A simple way to show this is with a scatter plot:

x-axis: CASE_COUNT_7DAY_AVG

y-axis: Subways: % of Comparable Pre-Pandemic Day

If the hypothesis is correct, I expect a downward-sloping pattern: higher case counts associated with lower subway ridership as a percentage of normal.


In [8]:
fig = px.scatter(
    combined,
    x="CASE_COUNT_7DAY_AVG",
    y="Subways: % of Comparable Pre-Pandemic Day",
    labels={
        "CASE_COUNT_7DAY_AVG": "COVID-19 Cases (7-Day Average)",
        "Subways: % of Comparable Pre-Pandemic Day": "Subway Ridership (% of Pre-Pandemic)"
    },
    title="Relationship Between COVID-19 Cases and Subway Ridership in NYC",
    opacity=0.4
)

# Wherever you have fig.show() replace it with this code:
HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))

This scatter plot shows a dense cloud of points, but the general trend is clear:

When CASE_COUNT_7DAY_AVG is very low (for example, in late 2021 or mid-2022), subway ridership as a percentage of pre-pandemic levels tends to be higher.

During waves where case counts spike sharply (for example, early 2020 or winter surges), subway ridership often falls to much lower percentages.

To quantify the direction of this relationship, we can compute the correlation coefficient:

In [9]:
combined[[
    "CASE_COUNT_7DAY_AVG",
    "Subways: % of Comparable Pre-Pandemic Day"
]].corr()


,CASE_COUNT_7DAY_AVG,Subways: % of Comparable Pre-Pandemic Day
CASE_COUNT_7DAY_AVG,1.000000,-0.156555
Subways: % of Comparable Pre-Pandemic Day,-0.156555,1.000000


The correlation is typically negative, consistent with the idea that as COVID-19 cases rise, subway usage declines.

6. Relationship 2: COVID-19 Cases and Bus Ridership

Next, I turn to the relationship between COVID-19 cases and bus ridership.

I repeat the same procedure as above, but with bus percentages:

In [10]:
fig = px.scatter(
    combined,
    x="CASE_COUNT_7DAY_AVG",
    y="Buses: % of Comparable Pre-Pandemic Day",
    labels={
        "CASE_COUNT_7DAY_AVG": "COVID-19 Cases (7-Day Average)",
        "Buses: % of Comparable Pre-Pandemic Day": "Bus Ridership (% of Pre-Pandemic)"
    },
    title="Relationship Between COVID-19 Cases and Bus Ridership in NYC",
    opacity=0.4
)

# Wherever you have fig.show() replace it with this code:
HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))


This second scatter plot shows a broadly similar pattern: higher case counts tend to coincide with lower bus ridership relative to pre-pandemic levels. However, the cloud may look somewhat different from the subway plot:

In some time periods, buses may retain slightly more of their pre-pandemic ridership than subways, possibly because they serve neighborhoods with fewer alternatives or shorter trips.

At very high case levels, both subway and bus usage drop sharply, reflecting citywide shutdowns and behavioral changes.

Again, I can check the correlation:

In [11]:
combined[[
    "CASE_COUNT_7DAY_AVG",
    "Buses: % of Comparable Pre-Pandemic Day"
]].corr()


,CASE_COUNT_7DAY_AVG,Buses: % of Comparable Pre-Pandemic Day
CASE_COUNT_7DAY_AVG,1.000000,-0.047574
Buses: % of Comparable Pre-Pandemic Day,-0.047574,1.000000


7. Takeaways

The analysis shows a clear negative relationship between COVID-19 case levels and public transit use in New York City, but subway and bus systems responded very differently to pandemic conditions. Subway ridership proved far more sensitive to changes in case counts, collapsing to below 10% of pre-pandemic levels during major waves and fluctuating widely as citywide risk perceptions shifted. Bus ridership, in contrast, remained noticeably more stable, rarely falling below 40–70% even at the height of surges. This pattern suggests that buses served populations with fewer alternatives—including essential workers, outer-borough commuters, and riders without access to private vehicles—while subway usage reflected greater flexibility to avoid travel and stronger behavioral reactions to perceived infection risk in crowded, enclosed environments. Overall, both systems show reduced ridership when COVID-19 cases rise, but the magnitude of the decline is dramatically steeper for subway riders, highlighting important differences in who depends on each mode of transportation and how mobility changed across the pandemic.

